### 💡 Silhouette Score

실루엣 스코어란 데이터가 자신이 속한 군집의 다른 데이터들과 얼마나 가깝고, 다른 군집의 데이터들과 얼마나 떨어져 있는지를 기준으로 평가한다.

좋은 군집화 결과는 다음 두가지 조건을 만족해야한다.
1. **응집도** : 같은 군집에 속한 데이터끼리는 가까워야 한다.
2. **분리도** : 서로 다른 군집에 속한 데이터끼리는 멀어야 한다.

> 실루엣 스코어는 군집화가 얼마나 잘 되었는지를 평가할 수 있다.

#### 1️⃣ a와 b
실루엣 스코어를 이해하기 위해 다음 두 가지 거리를 알면 된다.
- a: 자신이 속한 군집 내부의 평균 거리
- b: 가장 가까운 다른 군집과의 평균 거리

예를들어

| 계산 대상 | 평균 거리 |
| --- | --- |
| 자신이 속한 Cluster 0 | 2 |
| 다른 군집 Cluster 1 | 8 |
| 다른 군집 Cluster 2 | 10 |

이 경우 a는 2, b는 8이 된다.

❗️ 중심점과의 거리가 아니라 각 군집에 속한 데이터들과의 평균거리를 의미한다.

#### 2️⃣ silhouette socre 공식
$s=\frac{b-a}{\max(a,b)}$

앞에 예시에서 a=2, b=8이므로, 계산해보면 0.75가 나온다.
> a는 작을수록 좋고, b는 클수록 좋다

실루엣 스코어의 범위는 -1 ~ 1까지 나오게 되며, 1에 가까울수록 자신의 군집과는 가깝고 다른 군집에서 멀리 떨어져있다.

-1에 가까울수록 자신의 군집보다 다른 군집에 더 가까이 위치한다.

In [17]:
# 실습
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_samples

X, y = make_blobs(
    n_samples=500,
    centers=4,
    cluster_std=0.8,
    random_state=42
)

# 모델 생성 및 학습
model = KMeans(
    n_clusters=4,
    n_init=10,
    random_state=42
)

model.fit(X)

# silhouette_score() : 전체 데이터의 평균 실루엣 점수를 반환한다.
score = silhouette_score(X, model.labels_)
print(score)

# silhouette_samples() : 전체 평균이 아니라 개별 데이터의 실루엣 점수를 반환한다.
sample_scores = silhouette_samples(X, model.labels_)
print(sample_scores[:5])

0.8336135403234282
[0.76833427 0.84320636 0.83369362 0.74515079 0.89873344]


In [12]:
# k값에 따른 silhouette score 점수 차이를 데이터 프레임으로 확인
scores = []

k_values = range(2, 11)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    model.fit(X)

    scores.append(silhouette_score(X, model.labels_))

df = pd.DataFrame({
    "K" : k_values,
    "scores" : scores
})

df

,K,scores
0,2,0.607231
1,3,0.781919
2,4,0.833614
3,5,0.700468
4,6,0.563530
5,7,0.450574
6,8,0.348880
7,9,0.346955
8,10,0.350867


In [28]:
# 군집별 실루엣 스코어 분석
# 전체 평균의 실루엣 스코어가 좋을 순 있어도 특정 군집의 실루엣 스코어는 낮을 수 있다.

model = KMeans(
    n_clusters=4,
    n_init=10,
    random_state=42
)

model.fit(X)

sample_scores = silhouette_samples(X, model.labels_)

cluster_df = pd.DataFrame(
    X,
    columns=["feature1", "feature2"]
)

cluster_df["cluster"] = model.labels_
cluster_df["silhouette"] = sample_scores

# 군집별 평균 점수 출력
print(cluster_df.groupby("cluster")["silhouette"].mean())

cluster
0    0.862281
1    0.789140
2    0.899740
3    0.783294
Name: silhouette, dtype: float64


---
#### 실습 문제

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_samples

X, y = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=0.8,
    random_state=42
)

# 1. n_clusters=4인 KMeans 모델을 생성하고
# X를 이용하여 학습하기
# 변수명: model
model = KMeans(
    n_clusters=4,
    n_init=10,
    random_state=42
)
model.fit(X)

# 2. model.labels_를 labels 변수에 저장하기
labels = model.labels_

# 3. X와 labels를 사용하여 전체 Silhouette Score 계산
# 변수명: score
# silhouette_score() 활용
score = silhouette_score(X, labels)


# 4. X와 labels를 사용하여 개별 데이터의 점수 계산
# 변수명: sample_scores
# silhouette_samples() 활용
sample_scores = silhouette_samples(X, labels)

# 5. sample_scores의 처음 10개 값 출력
print(sample_scores[:10])

# 6. sample_scores의 shape 출력
print(sample_scores.shape)

# 7. sample_scores의 평균값 출력
print(sample_scores.mean())

# 8. score와 sample_scores의 평균값이 동일한지 확인
# np.isclose() 활용
np.isclose(score, sample_scores.mean())

# 9. K=2부터 K=10까지의 값을 k_values에 저장
# range() 활용
k_values = range(2, 11)

# 10. Silhouette Score를 저장할 빈 리스트 생성
# 변수명: scores
scores = []

# 11. k_values의 각 K에 대해 KMeans 모델을 생성하고
# X를 이용하여 학습하기
# 12. 11번 반복문 내부에서 각 모델의
# 전체 Silhouette Score를 계산하고 scores에 추가하기

for k in k_values:
    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    model.fit(X)

    scores.append(silhouette_score(X, model.labels_))

# 13. k_values와 scores를 이용하여 DataFrame 생성
# 변수명: df
# 컬럼명: K, Silhouette_Score
# 총 9개의 행이 생성되어야 함
df = pd.DataFrame({
    "K" : k_values,
    "Silhouette_Score" : scores
})

# 14. df 전체 출력
print(df)

# 16. df에서 Silhouette_Score가 가장 높은 행의
# 인덱스 라벨 확인
# idxmax() 활용
best_idx = df["Silhouette_Score"].idxmax()

# 17. 16번에서 구한 인덱스를 이용해
# 해당 행의 K 값과 Silhouette_Score 값 출력
# 변수명: best_k, best_score
best_k = df.loc[best_idx, "K"]
best_score = df.loc[best_idx, "Silhouette_Score"]
print(best_k)
print(best_score)

# 18. n_clusters=best_k인 최종 모델 생성 및 학습
# 변수명: final_model
final_model = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=42
)

final_model.fit(X)

# 19. final_model.labels_를 final_labels에 저장
final_labels = final_model.labels_

# 20. X와 final_labels를 이용하여
# 600개 데이터 각각의 Silhouette Score 계산
# 변수명: final_sample_scores
final_sample_scores = silhouette_samples(X, final_model.labels_)

# 21. X를 이용하여 DataFrame 생성
# 변수명: cluster_df
# 컬럼명: feature1, feature2
cluster_df = pd.DataFrame(X, columns=["feature1", "feature2"])

# 22. cluster_df에 cluster 컬럼 추가
# 값: final_labels
cluster_df["cluster"] = final_labels

# 23. cluster_df에 silhouette 컬럼 추가
# 값: final_sample_scores
cluster_df["silhouette"] = final_sample_scores

[0.8992363  0.92720976 0.8650754  0.84650498 0.89747586 0.90332413
 0.82447298 0.82788329 0.82922168 0.9152989 ]
(600,)
0.8309524574533401
    K  Silhouette_Score
0   2          0.603730
1   3          0.778429
2   4          0.830952
3   5          0.722111
4   6          0.583107
5   7          0.445647
6   8          0.337834
7   9          0.350873
8  10          0.342164
4
0.8309524574533401


,feature1,feature2,cluster
silhouette,,,
0.256932,-5.102211,8.194776,0.0
0.515270,-4.604994,9.671808,0.0
0.582815,-6.915595,7.277428,3.0
0.598556,-8.020271,9.275325,3.0
0.607128,-2.097159,12.096471,0.0
...,...,...,...
0.929989,-6.710813,-6.957480,1.0
0.930139,-6.788245,-6.759868,1.0
0.930167,-6.726748,-6.842960,1.0
